In [1]:
import pandas as pd
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

DATA_PATH = "tiktok.csv"
TEXT_COLUMN = "Comment"

# Load and cleaning dataset
df = pd.read_csv(DATA_PATH)
print(df.info())
df = df.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)
print(f"Loaded {len(df)} comments")
print(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 910 entries, 0 to 909
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   User     910 non-null    object
 1   Comment  885 non-null    object
 2   Date     910 non-null    object
 3   Reply    910 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 28.6+ KB
None
Loaded 885 comments
                                    User  \
0  https://www.tiktok.com/@rubegoldburgh   
1     https://www.tiktok.com/@bmaperkins   
2   https://www.tiktok.com/@john.boy2836   
3      https://www.tiktok.com/@anera5703   
4      https://www.tiktok.com/@ssviperfl   

                                             Comment    Date  Reply  
0    Why the fuck is the mayo in the ketchup bottle?  Jun-15   7357  
1  For the love of God, leave the egg off the burger  Jun-15   1655  
2                                        It’s giving  Jul-14    164  
3                                 Worse burger ever. 

In [2]:
# Short/near-empty comments add noise to topic modeling more than they help,

MIN_WORDS = 5
comments = df[TEXT_COLUMN].tolist()
mask = df[TEXT_COLUMN].str.split().str.len() >= MIN_WORDS
comments = df.loc[mask, TEXT_COLUMN].tolist()
print(f"Using {len(comments)} comments after short-text filtering")

Using 493 comments after short-text filtering


In [3]:
# Vectorizer controls the words shown per topic (stopword removal, n-grams).
# BERTopic itself still embeds full comments; this only shapes the topic labels.
vectorizer_model = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
)

topic_model = BERTopic(
    vectorizer_model=vectorizer_model,
    min_topic_size=4,      # lower than default since this dataset is small (~900 comments)
    calculate_probabilities=True,
    verbose=False,
)

topics, probs = topic_model.fit_transform(comments)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
# Inspect discovered topics: each row is one topic, with its top keywords
# and how many comments were assigned to it. Topic -1 is the "outlier" bucket
# (comments that didn't fit any cluster) — check this isn't too large.
topic_info = topic_model.get_topic_info()
print(topic_info)

    Topic  Count                                               Name  \
0      -1     75                               -1_mayo_add_bro_love   
1       0     48                       0_egg burger_egg_eggs_burger   
2       1     35                              1_egg_yolk_runny_lost   
3       2     29                   2_tomato_salt_salt pepper_pepper   
4       3     27            3_lettuce_lettuce tomato_veggies_tomato   
5       4     27               4_cheese_cheeseburger_american_slice   
6       5     26              5_mustard_mustard burger_sandwich_hot   
7       6     21          6_burger_perfect burger_burger dont_thats   
8       7     19             7_krabby_krabby patty_patty_looks like   
9       8     19                            8_pickles_cold_hot_heat   
10      9     19                  9_onion_egg onion_onion ring_ring   
11     10     18              10_ketchup_mayo ketchup_bottle_wheres   
12     11     17                                 11_20_come_10_mayo   
13    

In [5]:
# Attach the discovered topic + confidence back onto each comment,
# same output shape as the aspect/label/score table from project 02.
bertopic_df = pd.DataFrame({
    "comment": comments,
    "topic_id": topics,
    "topic_label": [topic_model.get_topic_info(t)["Name"].values[0] if t != -1 else "outlier" for t in topics],
    "confidence": [round(max(p), 3) if hasattr(p, "__len__") else round(p, 3) for p in probs],
})
print(bertopic_df.tail())

                                               comment  topic_id  \
488         You ruined it with the mustard and pickles        17   
489  Your ingredients are so well behaved and coope...        -1   
490                      You're supposed to use butter        -1   
491  YUCK TOO MUCH MUSTARD BRO THAT IS NOT A FREAKI...         5   
492  Yummmmmm I feel like they learned how to make ...        -1   

                                   topic_label  confidence  
488  17_pickles_pickle_mustard_mustard pickles       1.000  
489                                    outlier       0.010  
490                                    outlier       0.025  
491      5_mustard_mustard burger_sandwich_hot       1.000  
492                                    outlier       0.024  


In [6]:
# Save results
bertopic_df.to_csv("bertopic_results.csv", index=False)
topic_info.to_csv("bertopic_topic_summary.csv", index=False)